In [1]:
import os
import random
import sys
import numpy as np
import pandas as pd



In [2]:
SEED = 31
TRIALS = 1000  # keep trials as-is; focus on correctness/dtype alignment

TARGET = "scalar_coupling_constant"
PREDICTION = "pred"

DATA_DIR_CANDIDATES = [
    "/kaggle/data/champs-scalar-coupling",
    "/kaggle/input/champs-scalar-coupling",
    "/kaggle/data/input/champs-scalar-coupling",
    "/kaggle/input",
    "/kaggle/data",
]


def find_data_dir():
    for d in DATA_DIR_CANDIDATES:
        if os.path.exists(os.path.join(d, "train.csv")) and os.path.exists(
            os.path.join(d, "test.csv")
        ):
            return d
    if os.path.exists("train.csv") and os.path.exists("test.csv"):
        return "."
    raise FileNotFoundError(
        "Could not find train.csv/test.csv in expected Kaggle paths. "
        f"Tried: {DATA_DIR_CANDIDATES} and current directory."
    )


DATA_DIR = find_data_dir()
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
STRUCTURES_PATH = os.path.join(DATA_DIR, "structures.csv")
MULLIKEN_PATH = os.path.join(DATA_DIR, "mulliken_charges.csv")
SHIELD_PATH = os.path.join(DATA_DIR, "magnetic_shielding_tensors.csv")

print("Using DATA_DIR =", DATA_DIR)
print("TRAIN_PATH =", TRAIN_PATH)
print("TEST_PATH  =", TEST_PATH)
print(
    "STRUCTURES_PATH =", STRUCTURES_PATH, "| exists:", os.path.exists(STRUCTURES_PATH)
)
print("MULLIKEN_PATH   =", MULLIKEN_PATH, "| exists:", os.path.exists(MULLIKEN_PATH))
print("SHIELD_PATH     =", SHIELD_PATH, "| exists:", os.path.exists(SHIELD_PATH))




Using DATA_DIR = /kaggle/data/champs-scalar-coupling
TRAIN_PATH = /kaggle/data/champs-scalar-coupling/train.csv
TEST_PATH  = /kaggle/data/champs-scalar-coupling/test.csv
STRUCTURES_PATH = /kaggle/data/champs-scalar-coupling/structures.csv | exists: True
MULLIKEN_PATH   = /kaggle/data/champs-scalar-coupling/mulliken_charges.csv | exists: True
SHIELD_PATH     = /kaggle/data/champs-scalar-coupling/magnetic_shielding_tensors.csv | exists: True


In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(SEED)




In [4]:
# Change (score-relevant correctness): build type labels as stable plain-object strings
# directly from the raw 'type' column (NOT from Categorical), to prevent mapping/index
# mismatches that cause many rows to fall back to weak global/type means.
def make_type_lbl(s: pd.Series) -> pd.Series:
    s2 = s.astype(object)
    mask = pd.notna(s2)
    out = pd.Series(np.nan, index=s.index, dtype=object)
    out.loc[mask] = s2.loc[mask].map(lambda v: str(v))
    return out


def group_mean_log_mae(y_true, y_pred, types_lbl, floor=1e-9):
    maes = (y_true - y_pred).abs().groupby(types_lbl).mean()
    maes = np.log(maes.clip(lower=floor))
    return float(maes.mean())




In [5]:
usecols_train = ["id", "molecule_name", "atom_index_0", "atom_index_1", "type", TARGET]
usecols_test = ["id", "molecule_name", "atom_index_0", "atom_index_1", "type"]

dtype_train = {
    "id": np.int32,
    "molecule_name": "object",
    "atom_index_0": np.int32,
    "atom_index_1": np.int32,
    "type": "object",
    TARGET: np.float32,
}
dtype_test = {
    "id": np.int32,
    "molecule_name": "object",
    "atom_index_0": np.int32,
    "atom_index_1": np.int32,
    "type": "object",
}

train = pd.read_csv(TRAIN_PATH, usecols=usecols_train, dtype=dtype_train)
test = pd.read_csv(TEST_PATH, usecols=usecols_test, dtype=dtype_test)

# Change (score-relevant correctness): create type_lbl from the raw object 'type' values
# before any categoricals, and then ensure 'type' is categorical only for compactness.
train["type_lbl"] = make_type_lbl(train["type"])
test["type_lbl"] = make_type_lbl(test["type"])

# Hard guardrail: any missing type_lbl would break all type-wise models and inflate error.
if train["type_lbl"].isna().any() or test["type_lbl"].isna().any():
    raise ValueError("Found missing type_lbl values; cannot proceed safely.")

all_types = pd.Index(pd.concat([train["type_lbl"], test["type_lbl"]], axis=0).unique())
train["type"] = pd.Categorical(train["type_lbl"], categories=all_types)
test["type"] = pd.Categorical(test["type_lbl"], categories=all_types)

print("Loaded:", train.shape, test.shape)
print("Unique molecules (train):", train["molecule_name"].nunique())
print("Unique types:", pd.Series(train["type_lbl"]).nunique())
print(
    "atom_index_0 max/min:",
    int(train["atom_index_0"].max()),
    int(train["atom_index_0"].min()),
)




Loaded: (4191263, 7) (467813, 6)
Unique molecules (train): 76510
Unique types: 8
atom_index_0 max/min: 28 1


In [6]:
def molecule_holdout_split(df, valid_frac=0.10, seed=SEED):
    mols = df["molecule_name"].unique()
    rng = np.random.RandomState(seed)
    rng.shuffle(mols)
    n_valid = max(1, int(len(mols) * valid_frac))
    valid_mols = set(mols[:n_valid])
    is_valid = df["molecule_name"].isin(valid_mols)
    return df[~is_valid].copy(), df[is_valid].copy()


train_tr, train_va = molecule_holdout_split(train, valid_frac=0.10, seed=SEED)
train_tr2, train_va2 = molecule_holdout_split(train, valid_frac=0.10, seed=SEED + 997)

print("Fold1 Train:", train_tr.shape, "Valid:", train_va.shape)
print("Fold2 Train:", train_tr2.shape, "Valid:", train_va2.shape)




Fold1 Train: (3771392, 7) Valid: (419871, 7)
Fold2 Train: (3770584, 7) Valid: (420679, 7)


In [7]:
def build_type_mappings(train_df, k1=25.0, k2=100.0):
    t = train_df["type_lbl"]
    type_mean = train_df[TARGET].groupby(t).mean()
    type_median = train_df[TARGET].groupby(t).median()
    global_mean = float(train_df[TARGET].mean())

    type_count = train_df[TARGET].groupby(t).size().astype(float)
    shrink_mean_k1 = (type_mean * type_count + global_mean * k1) / (type_count + k1)
    shrink_mean_k2 = (type_mean * type_count + global_mean * k2) / (type_count + k2)

    return {
        "type_mean": type_mean,
        "type_median": type_median,
        "shrink_k1": shrink_mean_k1,
        "shrink_k2": shrink_mean_k2,
        "global_mean": global_mean,
    }


def build_pred_df(base, mapping, global_mean):
    df = base[["id", "type", "type_lbl"]].copy()
    df[PREDICTION] = df["type_lbl"].map(mapping).astype(float)
    df[PREDICTION] = df[PREDICTION].fillna(global_mean)
    return df


def build_train_pred_df(base, mapping, global_mean):
    df = base[["id", "type", "type_lbl", TARGET]].copy()
    df[PREDICTION] = df["type_lbl"].map(mapping).astype(float)
    df[PREDICTION] = df[PREDICTION].fillna(global_mean)
    return df




In [8]:
def _read_csv_in_chunks_filter_mols(
    path, usecols, mol_names, chunksize=250_000, dtypes=None
):
    mol_names = set(mol_names)
    out = []
    reader = pd.read_csv(path, usecols=usecols, chunksize=chunksize, dtype=dtypes)
    for ch in reader:
        ch = ch[ch["molecule_name"].isin(mol_names)]
        if len(ch):
            out.append(ch)
    if not out:
        return pd.DataFrame(columns=usecols)
    return pd.concat(out, axis=0, ignore_index=True)


def load_structures_needed(structures_path, mol_names):
    usecols = ["molecule_name", "atom_index", "atom", "x", "y", "z"]
    dtypes = {
        "molecule_name": "object",
        "atom_index": np.int32,  # int32 to prevent overflow and merge failure
        "atom": "object",
        "x": np.float32,
        "y": np.float32,
        "z": np.float32,
    }
    st = _read_csv_in_chunks_filter_mols(
        structures_path, usecols, mol_names, dtypes=dtypes
    )
    return st


def attach_atoms_and_distance(df, structures_df):
    df = df.copy()
    df["molecule_name"] = df["molecule_name"].astype(object)
    df["atom_index_0"] = df["atom_index_0"].astype(np.int32)
    df["atom_index_1"] = df["atom_index_1"].astype(np.int32)

    structures_df = structures_df.copy()
    structures_df["molecule_name"] = structures_df["molecule_name"].astype(object)
    structures_df["atom_index"] = structures_df["atom_index"].astype(np.int32)

    s0 = structures_df.rename(
        columns={
            "atom_index": "atom_index_0",
            "atom": "atom_0",
            "x": "x0",
            "y": "y0",
            "z": "z0",
        }
    )
    out = df.merge(
        s0[["molecule_name", "atom_index_0", "atom_0", "x0", "y0", "z0"]],
        on=["molecule_name", "atom_index_0"],
        how="left",
    )

    s1 = structures_df.rename(
        columns={
            "atom_index": "atom_index_1",
            "atom": "atom_1",
            "x": "x1",
            "y": "y1",
            "z": "z1",
        }
    )
    out = out.merge(
        s1[["molecule_name", "atom_index_1", "atom_1", "x1", "y1", "z1"]],
        on=["molecule_name", "atom_index_1"],
        how="left",
    )

    miss0 = out[["x0", "y0", "z0"]].isna().any(axis=1).mean()
    miss1 = out[["x1", "y1", "z1"]].isna().any(axis=1).mean()
    if (miss0 > 0.001) or (miss1 > 0.001):
        print(
            f"[WARN] structures join missing-rate: atom0={miss0:.4%}, atom1={miss1:.4%}. "
            "Filling missing coords with 0.0 to avoid catastrophic NaNs."
        )
        for c in ["x0", "y0", "z0", "x1", "y1", "z1"]:
            out[c] = out[c].astype(np.float32).fillna(0.0)

    dx = out["x0"].to_numpy(dtype=np.float64) - out["x1"].to_numpy(dtype=np.float64)
    dy = out["y0"].to_numpy(dtype=np.float64) - out["y1"].to_numpy(dtype=np.float64)
    dz = out["z0"].to_numpy(dtype=np.float64) - out["z1"].to_numpy(dtype=np.float64)
    out["dist"] = np.sqrt(dx * dx + dy * dy + dz * dz).astype(np.float32)

    a0 = out["atom_0"].astype(str)
    a1 = out["atom_1"].astype(str)
    out["atom_pair"] = np.where(a0 <= a1, a0 + "_" + a1, a1 + "_" + a0)

    return out


def winsorize_dist_per_type(train_ref_df, df_to_clip, lo_q=0.01, hi_q=0.99):
    qs = train_ref_df.groupby("type_lbl")["dist"].quantile([lo_q, hi_q]).unstack()
    qs.columns = ["lo", "hi"]
    out = df_to_clip.copy()
    lo = out["type_lbl"].map(qs["lo"]).astype(float)
    hi = out["type_lbl"].map(qs["hi"]).astype(float)
    lo = lo.fillna(float(train_ref_df["dist"].quantile(lo_q)))
    hi = hi.fillna(float(train_ref_df["dist"].quantile(hi_q)))
    out["dist"] = np.clip(
        out["dist"].astype(float).to_numpy(), lo.to_numpy(), hi.to_numpy()
    ).astype(np.float32)
    return out


fold1_mols = set(
    pd.concat([train_tr["molecule_name"], train_va["molecule_name"]]).unique()
)
fold2_mols = set(
    pd.concat([train_tr2["molecule_name"], train_va2["molecule_name"]]).unique()
)
final_train_mols = set(train["molecule_name"].unique())
final_test_mols = set(test["molecule_name"].unique())

structures_f1 = load_structures_needed(STRUCTURES_PATH, fold1_mols)
structures_f2 = load_structures_needed(STRUCTURES_PATH, fold2_mols)
structures_train = load_structures_needed(STRUCTURES_PATH, final_train_mols)
structures_test = load_structures_needed(STRUCTURES_PATH, final_test_mols)

print("Loaded structures fold1:", structures_f1.shape)
print("Loaded structures fold2:", structures_f2.shape)
print("Loaded structures train:", structures_train.shape)
print("Loaded structures test :", structures_test.shape)

train_tr_f = attach_atoms_and_distance(train_tr, structures_f1)
train_va_f = attach_atoms_and_distance(train_va, structures_f1)

train_tr2_f = attach_atoms_and_distance(train_tr2, structures_f2)
train_va2_f = attach_atoms_and_distance(train_va2, structures_f2)

train_f = attach_atoms_and_distance(train, structures_train)
test_f = attach_atoms_and_distance(test, structures_test)

for df_name, df_ in [
    ("train_tr_f", train_tr_f),
    ("train_va_f", train_va_f),
    ("train_tr2_f", train_tr2_f),
    ("train_va2_f", train_va2_f),
    ("train_f", train_f),
    ("test_f", test_f),
]:
    if df_["dist"].isna().any():
        df_["dist"] = df_["dist"].fillna(float(np.nanmedian(df_["dist"].to_numpy())))
        print(df_name, "had missing dist; filled with median.")

train_tr_f = winsorize_dist_per_type(train_tr_f, train_tr_f, 0.01, 0.99)
train_va_f = winsorize_dist_per_type(train_tr_f, train_va_f, 0.01, 0.99)

train_tr2_f = winsorize_dist_per_type(train_tr2_f, train_tr2_f, 0.01, 0.99)
train_va2_f = winsorize_dist_per_type(train_tr2_f, train_va2_f, 0.01, 0.99)

train_f = winsorize_dist_per_type(train_f, train_f, 0.01, 0.99)
test_f = winsorize_dist_per_type(train_f, test_f, 0.01, 0.99)




Loaded structures fold1: (1379964, 6)
Loaded structures fold2: (1379964, 6)
Loaded structures train: (1379964, 6)
Loaded structures test : (0, 6)
[WARN] structures join missing-rate: atom0=100.0000%, atom1=100.0000%. Filling missing coords with 0.0 to avoid catastrophic NaNs.


In [9]:
def build_atompair_mean_mapping(train_df, k_pair=50.0):
    global_mean = float(train_df[TARGET].mean())
    type_mean = train_df.groupby("type_lbl")[TARGET].mean()

    tmp = train_df.copy()
    pair_mean = tmp.groupby(["type_lbl", "atom_pair"])[TARGET].mean()
    pair_cnt = tmp.groupby(["type_lbl", "atom_pair"])[TARGET].size().astype(float)

    type_mean_for_pair = (
        pair_mean.index.get_level_values(0).map(type_mean).astype(float)
    )
    pair_shrunk = (pair_mean * pair_cnt + type_mean_for_pair * k_pair) / (
        pair_cnt + k_pair
    )
    pair_shrunk.index = pair_shrunk.index.set_names(["type_lbl", "atom_pair"])
    return global_mean, type_mean, pair_shrunk


def predict_atompair_mean(df, global_mean, type_mean, pair_mean):
    out = df[["id", "type", "type_lbl"]].copy()
    idx = pd.MultiIndex.from_arrays(
        [
            df["type_lbl"].astype(object).to_numpy(),
            df["atom_pair"].astype(str).to_numpy(),
        ],
        names=["type_lbl", "atom_pair"],
    )
    vals = pair_mean.reindex(idx).to_numpy()
    tmean = df["type_lbl"].map(type_mean).astype(float).to_numpy()
    pred = np.where(np.isfinite(vals), vals, tmean)
    pred = np.where(np.isfinite(pred), pred, global_mean)
    out[PREDICTION] = pred.astype(float)
    return out


def fit_typewise_dist_linear(train_df, ridge=1e-3):
    global_mean = float(train_df[TARGET].mean())
    type_mean = train_df.groupby("type_lbl")[TARGET].mean()

    params = {}
    for t, g in train_df.groupby("type_lbl"):
        x = g["dist"].astype(float).to_numpy()
        y = g[TARGET].astype(float).to_numpy()
        if len(x) < 2 or not np.isfinite(x).all():
            params[t] = (float(type_mean.loc[t]), 0.0)
            continue
        x_mean = float(x.mean())
        y_mean = float(y.mean())
        denom = float(((x - x_mean) ** 2).sum()) + float(ridge)
        b = float(((x - x_mean) * (y - y_mean)).sum() / denom)
        a = y_mean - b * x_mean
        params[t] = (a, b)
    return global_mean, type_mean, params


def predict_typewise_dist_linear(df, global_mean, type_mean, params):
    out = df[["id", "type", "type_lbl"]].copy()
    t = df["type_lbl"].astype(object)
    ab = t.map(params)
    a = np.array(
        [v[0] if isinstance(v, tuple) else np.nan for v in ab], dtype=np.float64
    )
    b = np.array(
        [v[1] if isinstance(v, tuple) else np.nan for v in ab], dtype=np.float64
    )

    a_missing = ~np.isfinite(a)
    if a_missing.any():
        a[a_missing] = t.map(type_mean).astype(float).to_numpy()[a_missing]
    b_missing = ~np.isfinite(b)
    if b_missing.any():
        b[b_missing] = 0.0

    x = df["dist"].astype(float).to_numpy()
    pred = a + b * x
    pred = np.where(
        np.isfinite(pred), pred, df["type_lbl"].map(type_mean).astype(float).to_numpy()
    )
    pred = np.where(np.isfinite(pred), pred, global_mean)
    out[PREDICTION] = pred.astype(float)
    return out


def load_mulliken_needed(path, mol_names):
    usecols = ["molecule_name", "atom_index", "mulliken_charge"]
    dtypes = {
        "molecule_name": "object",
        "atom_index": np.int32,
        "mulliken_charge": np.float32,
    }
    mc = _read_csv_in_chunks_filter_mols(path, usecols, mol_names, dtypes=dtypes)
    return mc


def load_shielding_needed(path, mol_names):
    usecols = [
        "molecule_name",
        "atom_index",
        "XX",
        "YX",
        "ZX",
        "XY",
        "YY",
        "ZY",
        "XZ",
        "YZ",
        "ZZ",
    ]
    dtypes = {
        c: np.float32 for c in ["XX", "YX", "ZX", "XY", "YY", "ZY", "XZ", "YZ", "ZZ"]
    }
    dtypes.update({"molecule_name": "object", "atom_index": np.int32})
    sh = _read_csv_in_chunks_filter_mols(path, usecols, mol_names, dtypes=dtypes)
    if len(sh):
        sh["shield_trace"] = (sh["XX"] + sh["YY"] + sh["ZZ"]).astype(np.float32)
        sh["shield_abs_sum"] = (
            sh[["XX", "YX", "ZX", "XY", "YY", "ZY", "XZ", "YZ", "ZZ"]]
            .abs()
            .sum(axis=1)
            .astype(np.float32)
        )
        sh = sh[["molecule_name", "atom_index", "shield_trace", "shield_abs_sum"]]
    else:
        sh = pd.DataFrame(
            columns=["molecule_name", "atom_index", "shield_trace", "shield_abs_sum"]
        )
    return sh


def attach_mulliken(df, mulliken_df):
    df = df.copy()
    df["molecule_name"] = df["molecule_name"].astype(object)
    df["atom_index_0"] = df["atom_index_0"].astype(np.int32)
    df["atom_index_1"] = df["atom_index_1"].astype(np.int32)

    mulliken_df = mulliken_df.copy()
    mulliken_df["molecule_name"] = mulliken_df["molecule_name"].astype(object)
    mulliken_df["atom_index"] = mulliken_df["atom_index"].astype(np.int32)

    m0 = mulliken_df.rename(
        columns={"atom_index": "atom_index_0", "mulliken_charge": "q0"}
    )
    m1 = mulliken_df.rename(
        columns={"atom_index": "atom_index_1", "mulliken_charge": "q1"}
    )
    out = df.merge(
        m0[["molecule_name", "atom_index_0", "q0"]],
        on=["molecule_name", "atom_index_0"],
        how="left",
    )
    out = out.merge(
        m1[["molecule_name", "atom_index_1", "q1"]],
        on=["molecule_name", "atom_index_1"],
        how="left",
    )
    for c in ["q0", "q1"]:
        if c in out.columns:
            out[c] = out[c].astype(np.float32)
            out[c] = out[c].fillna(0.0)
    return out


def attach_shielding(df, shielding_df):
    df = df.copy()
    df["molecule_name"] = df["molecule_name"].astype(object)
    df["atom_index_0"] = df["atom_index_0"].astype(np.int32)
    df["atom_index_1"] = df["atom_index_1"].astype(np.int32)

    shielding_df = shielding_df.copy()
    shielding_df["molecule_name"] = shielding_df["molecule_name"].astype(object)
    shielding_df["atom_index"] = shielding_df["atom_index"].astype(np.int32)

    s0 = shielding_df.rename(
        columns={
            "atom_index": "atom_index_0",
            "shield_trace": "st0",
            "shield_abs_sum": "sa0",
        }
    )
    s1 = shielding_df.rename(
        columns={
            "atom_index": "atom_index_1",
            "shield_trace": "st1",
            "shield_abs_sum": "sa1",
        }
    )
    out = df.merge(
        s0[["molecule_name", "atom_index_0", "st0", "sa0"]],
        on=["molecule_name", "atom_index_0"],
        how="left",
    )
    out = out.merge(
        s1[["molecule_name", "atom_index_1", "st1", "sa1"]],
        on=["molecule_name", "atom_index_1"],
        how="left",
    )
    for c in ["st0", "sa0", "st1", "sa1"]:
        if c in out.columns:
            out[c] = out[c].astype(np.float32)
            out[c] = out[c].fillna(0.0)
    return out


def fit_typewise_linear_multi(train_df, feat_cols, ridge=1e-3):
    global_mean = float(train_df[TARGET].mean())
    type_mean = train_df.groupby("type_lbl")[TARGET].mean()
    params = {}

    for t, g in train_df.groupby("type_lbl"):
        X = g[feat_cols].astype(float).to_numpy()
        y = g[TARGET].astype(float).to_numpy()
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        if len(y) < (len(feat_cols) + 2):
            params[t] = (
                float(type_mean.loc[t]),
                np.zeros(len(feat_cols), dtype=np.float64),
            )
            continue
        X_mean = X.mean(axis=0, keepdims=True)
        y_mean = float(y.mean())
        Xc = X - X_mean
        yc = y - y_mean
        A = Xc.T @ Xc + ridge * np.eye(Xc.shape[1])
        b = Xc.T @ yc
        w = np.linalg.solve(A, b)
        intercept = y_mean - float((X_mean @ w.reshape(-1, 1)).ravel()[0])
        params[t] = (float(intercept), w.astype(np.float64))
    return global_mean, type_mean, params


def predict_typewise_linear_multi(df, feat_cols, global_mean, type_mean, params):
    out = df[["id", "type", "type_lbl"]].copy()
    X = df[feat_cols].astype(float).to_numpy()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    t = df["type_lbl"].astype(object)
    pw = t.map(params)
    a = np.array(
        [v[0] if isinstance(v, tuple) else np.nan for v in pw], dtype=np.float64
    )
    w_list = [v[1] if isinstance(v, tuple) else None for v in pw]

    a_missing = ~np.isfinite(a)
    if a_missing.any():
        a[a_missing] = t.map(type_mean).astype(float).to_numpy()[a_missing]

    pred = np.empty(len(df), dtype=np.float64)
    for i in range(len(df)):
        wi = w_list[i]
        if wi is None:
            pred[i] = a[i]
        else:
            pred[i] = a[i] + float(X[i].dot(wi))

    pred = np.where(
        np.isfinite(pred), pred, df["type_lbl"].map(type_mean).astype(float).to_numpy()
    )
    pred = np.where(np.isfinite(pred), pred, global_mean)
    out[PREDICTION] = pred.astype(float)
    return out




In [10]:
def build_valid_sets(
    train_tr_df,
    train_va_df,
    train_tr_f_df,
    train_va_f_df,
    want_mulliken=True,
    want_shield=True,
):
    maps_tr = build_type_mappings(train_tr_df, k1=25.0, k2=100.0)
    gm_tr = maps_tr["global_mean"]

    valid_sets_local = []
    model_names = []

    va_m0 = build_train_pred_df(train_va_df, maps_tr["type_mean"], gm_tr)
    valid_sets_local.append(va_m0)
    model_names.append("type_mean")

    va_m1 = build_train_pred_df(train_va_df, maps_tr["type_median"], gm_tr)
    valid_sets_local.append(va_m1)
    model_names.append("type_median")

    va_m2 = build_train_pred_df(train_va_df, maps_tr["shrink_k1"], gm_tr)
    valid_sets_local.append(va_m2)
    model_names.append("shrink_k1")

    va_m3 = build_train_pred_df(train_va_df, maps_tr["shrink_k2"], gm_tr)
    valid_sets_local.append(va_m3)
    model_names.append("shrink_k2")

    gmean_ap, tmean_ap, pair_mean = build_atompair_mean_mapping(
        train_tr_f_df, k_pair=50.0
    )
    va_m4 = train_va_f_df[["id", "type", "type_lbl", TARGET]].copy()
    va_m4[PREDICTION] = predict_atompair_mean(
        train_va_f_df, gmean_ap, tmean_ap, pair_mean
    )[PREDICTION].values
    valid_sets_local.append(va_m4)
    model_names.append("atompair_shrunk")

    gmean_lin, tmean_lin, params_lin = fit_typewise_dist_linear(
        train_tr_f_df, ridge=1e-3
    )
    va_m5 = train_va_f_df[["id", "type", "type_lbl", TARGET]].copy()
    va_m5[PREDICTION] = predict_typewise_dist_linear(
        train_va_f_df, gmean_lin, tmean_lin, params_lin
    )[PREDICTION].values
    valid_sets_local.append(va_m5)
    model_names.append("type_dist_linear")

    fold_mols = set(
        pd.concat([train_tr_df["molecule_name"], train_va_df["molecule_name"]]).unique()
    )

    if want_mulliken and os.path.exists(MULLIKEN_PATH):
        mulliken = load_mulliken_needed(MULLIKEN_PATH, fold_mols)
        train_tr_q = attach_mulliken(train_tr_f_df, mulliken)
        train_va_q = attach_mulliken(train_va_f_df, mulliken)

        for d in [train_tr_q, train_va_q]:
            d["q0"] = d["q0"].astype(np.float32)
            d["q1"] = d["q1"].astype(np.float32)
            d["qprod"] = (d["q0"] * d["q1"]).astype(np.float32)
            d["qdiff"] = (d["q0"] - d["q1"]).astype(np.float32)
            d["qdiff"] = d["qdiff"].abs().astype(np.float32)

        feat_q = ["dist", "q0", "q1", "qprod", "qdiff"]
        gmean_q, tmean_q, params_q = fit_typewise_linear_multi(
            train_tr_q, feat_q, ridge=1e-3
        )
        va_m6 = train_va_q[["id", "type", "type_lbl", TARGET]].copy()
        va_m6[PREDICTION] = predict_typewise_linear_multi(
            train_va_q, feat_q, gmean_q, tmean_q, params_q
        )[PREDICTION].values
        valid_sets_local.append(va_m6)
        model_names.append("mulliken_linear")

    if want_shield and os.path.exists(SHIELD_PATH):
        shield = load_shielding_needed(SHIELD_PATH, fold_mols)
        train_tr_s = attach_shielding(train_tr_f_df, shield)
        train_va_s = attach_shielding(train_va_f_df, shield)

        for d in [train_tr_s, train_va_s]:
            d["st0"] = d["st0"].astype(np.float32)
            d["st1"] = d["st1"].astype(np.float32)
            d["sa0"] = d["sa0"].astype(np.float32)
            d["sa1"] = d["sa1"].astype(np.float32)
            d["st_sum"] = (d["st0"] + d["st1"]).astype(np.float32)
            d["st_diff"] = (d["st0"] - d["st1"]).astype(np.float32)
            d["st_diff"] = d["st_diff"].abs().astype(np.float32)
            d["sa_sum"] = (d["sa0"] + d["sa1"]).astype(np.float32)

        feat_s = ["dist", "st0", "st1", "st_sum", "st_diff", "sa_sum"]
        gmean_s, tmean_s, params_s = fit_typewise_linear_multi(
            train_tr_s, feat_s, ridge=1e-3
        )
        va_m7 = train_va_s[["id", "type", "type_lbl", TARGET]].copy()
        va_m7[PREDICTION] = predict_typewise_linear_multi(
            train_va_s, feat_s, gmean_s, tmean_s, params_s
        )[PREDICTION].values
        valid_sets_local.append(va_m7)
        model_names.append("shield_linear")

    for i in range(len(valid_sets_local)):
        valid_sets_local[i] = (
            valid_sets_local[i].sort_values("id").reset_index(drop=True)
        )
    for i in range(1, len(valid_sets_local)):
        assert (
            valid_sets_local[0]["id"].values == valid_sets_local[i]["id"].values
        ).all()

    return valid_sets_local, model_names


USE_MULLIKEN = bool(os.path.exists(MULLIKEN_PATH))
USE_SHIELD = bool(os.path.exists(SHIELD_PATH))

valid_sets_fold1, names1 = build_valid_sets(
    train_tr, train_va, train_tr_f, train_va_f, USE_MULLIKEN, USE_SHIELD
)
valid_sets_fold2, names2 = build_valid_sets(
    train_tr2, train_va2, train_tr2_f, train_va2_f, USE_MULLIKEN, USE_SHIELD
)

print("Fold1 models:", names1)
print("Fold2 models:", names2)
print(
    "Prepared validation prediction sets fold1:", [df.shape for df in valid_sets_fold1]
)
print(
    "Prepared validation prediction sets fold2:", [df.shape for df in valid_sets_fold2]
)

common_len = min(len(valid_sets_fold1), len(valid_sets_fold2))
valid_sets_fold1 = valid_sets_fold1[:common_len]
valid_sets_fold2 = valid_sets_fold2[:common_len]
names1 = names1[:common_len]
names2 = names2[:common_len]
assert len(valid_sets_fold1) == len(valid_sets_fold2)




Fold1 models: ['type_mean', 'type_median', 'shrink_k1', 'shrink_k2', 'atompair_shrunk', 'type_dist_linear', 'mulliken_linear', 'shield_linear']
Fold2 models: ['type_mean', 'type_median', 'shrink_k1', 'shrink_k2', 'atompair_shrunk', 'type_dist_linear', 'mulliken_linear', 'shield_linear']
Prepared validation prediction sets fold1: [(419871, 5), (419871, 5), (419871, 5), (419871, 5), (419871, 5), (419871, 5), (419871, 5), (419871, 5)]
Prepared validation prediction sets fold2: [(420679, 5), (420679, 5), (420679, 5), (420679, 5), (420679, 5), (420679, 5), (420679, 5), (420679, 5)]


In [11]:
def weights(n, alpha=1.0):
    if n < 1:
        raise ValueError("n must not be less than 1")
    return np.random.dirichlet(alpha=np.full(n, alpha)).tolist()


def eval_blend(valid_sets, ws):
    df = valid_sets[0][["id", "type", "type_lbl", TARGET]].copy()
    df[PREDICTION] = 0.0
    for i, t in enumerate(valid_sets):
        df[PREDICTION] += t[PREDICTION].astype(float).values * ws[i]
    return float(group_mean_log_mae(df[TARGET], df[PREDICTION], df["type_lbl"]))


def trial_twofold(valid_sets1, valid_sets2):
    n = min(len(valid_sets1), len(valid_sets2))
    ws = weights(n, alpha=1.0)
    s1 = eval_blend(valid_sets1[:n], ws)
    s2 = eval_blend(valid_sets2[:n], ws)
    return 0.5 * (s1 + s2), ws, s1, s2


best = sys.maxsize
best_weights = None
best_s1 = None
best_s2 = None

for i in range(TRIALS):
    score, ws, s1, s2 = trial_twofold(valid_sets_fold1, valid_sets_fold2)
    if score < best:
        best = score
        best_weights = ws
        best_s1 = s1
        best_s2 = s2

print(f"best(avg valid)={best:.6f} | fold1={best_s1:.6f} fold2={best_s2:.6f}")
print(f"best weights (sum={sum(best_weights):.6f})")
for i, w in enumerate(best_weights):
    print(f"  model{i} ({names1[i] if i < len(names1) else 'model'}) weight={w:.6f}")



best(avg valid)=0.868068 | fold1=0.870894 fold2=0.865241
best weights (sum=1.000000)
  model0 (type_mean) weight=0.009235
  model1 (type_median) weight=0.078817
  model2 (shrink_k1) weight=0.043504
  model3 (shrink_k2) weight=0.018497
  model4 (atompair_shrunk) weight=0.030908
  model5 (type_dist_linear) weight=0.070500
  model6 (mulliken_linear) weight=0.549091
  model7 (shield_linear) weight=0.199447


In [12]:
maps_full = build_type_mappings(train, k1=25.0, k2=100.0)
global_mean = maps_full["global_mean"]

test_m0 = (
    build_pred_df(test, maps_full["type_mean"], global_mean)
    .sort_values("id")
    .reset_index(drop=True)
)
test_m1 = (
    build_pred_df(test, maps_full["type_median"], global_mean)
    .sort_values("id")
    .reset_index(drop=True)
)
test_m2 = (
    build_pred_df(test, maps_full["shrink_k1"], global_mean)
    .sort_values("id")
    .reset_index(drop=True)
)
test_m3 = (
    build_pred_df(test, maps_full["shrink_k2"], global_mean)
    .sort_values("id")
    .reset_index(drop=True)
)

gmean_ap_f, tmean_ap_f, pair_mean_f = build_atompair_mean_mapping(train_f, k_pair=50.0)
test_m4 = (
    predict_atompair_mean(test_f, gmean_ap_f, tmean_ap_f, pair_mean_f)
    .sort_values("id")
    .reset_index(drop=True)
)

gmean_lin_f, tmean_lin_f, params_lin_f = fit_typewise_dist_linear(train_f, ridge=1e-3)
test_m5 = (
    predict_typewise_dist_linear(test_f, gmean_lin_f, tmean_lin_f, params_lin_f)
    .sort_values("id")
    .reset_index(drop=True)
)

test_models = [test_m0, test_m1, test_m2, test_m3, test_m4, test_m5]
test_model_names = [
    "type_mean",
    "type_median",
    "shrink_k1",
    "shrink_k2",
    "atompair_shrunk",
    "type_dist_linear",
]

final_train_mols = set(train["molecule_name"].unique())
final_test_mols = set(test["molecule_name"].unique())

if USE_MULLIKEN and os.path.exists(MULLIKEN_PATH):
    mulliken_train = load_mulliken_needed(MULLIKEN_PATH, final_train_mols)
    mulliken_test = load_mulliken_needed(MULLIKEN_PATH, final_test_mols)
    train_q = attach_mulliken(train_f, mulliken_train)
    test_q = attach_mulliken(test_f, mulliken_test)
    for d in [train_q, test_q]:
        d["q0"] = d["q0"].astype(np.float32).fillna(0.0)
        d["q1"] = d["q1"].astype(np.float32).fillna(0.0)
        d["qprod"] = (d["q0"] * d["q1"]).astype(np.float32)
        d["qdiff"] = (d["q0"] - d["q1"]).astype(np.float32)
        d["qdiff"] = d["qdiff"].abs().astype(np.float32)
    feat_q = ["dist", "q0", "q1", "qprod", "qdiff"]
    gmean_q_f, tmean_q_f, params_q_f = fit_typewise_linear_multi(
        train_q, feat_q, ridge=1e-3
    )
    test_m6 = (
        predict_typewise_linear_multi(test_q, feat_q, gmean_q_f, tmean_q_f, params_q_f)
        .sort_values("id")
        .reset_index(drop=True)
    )
    test_models.append(test_m6)
    test_model_names.append("mulliken_linear")

if USE_SHIELD and os.path.exists(SHIELD_PATH):
    shield_train = load_shielding_needed(SHIELD_PATH, final_train_mols)
    shield_test = load_shielding_needed(SHIELD_PATH, final_test_mols)
    train_s = attach_shielding(train_f, shield_train)
    test_s = attach_shielding(test_f, shield_test)
    for d in [train_s, test_s]:
        d["st0"] = d["st0"].astype(np.float32).fillna(0.0)
        d["st1"] = d["st1"].astype(np.float32).fillna(0.0)
        d["sa0"] = d["sa0"].astype(np.float32).fillna(0.0)
        d["sa1"] = d["sa1"].astype(np.float32).fillna(0.0)
        d["st_sum"] = (d["st0"] + d["st1"]).astype(np.float32)
        d["st_diff"] = (d["st0"] - d["st1"]).astype(np.float32)
        d["st_diff"] = d["st_diff"].abs().astype(np.float32)
        d["sa_sum"] = (d["sa0"] + d["sa1"]).astype(np.float32)
    feat_s = ["dist", "st0", "st1", "st_sum", "st_diff", "sa_sum"]
    gmean_s_f, tmean_s_f, params_s_f = fit_typewise_linear_multi(
        train_s, feat_s, ridge=1e-3
    )
    test_m7 = (
        predict_typewise_linear_multi(test_s, feat_s, gmean_s_f, tmean_s_f, params_s_f)
        .sort_values("id")
        .reset_index(drop=True)
    )
    test_models.append(test_m7)
    test_model_names.append("shield_linear")

for i in range(1, len(test_models)):
    assert (test_models[0]["id"].values == test_models[i]["id"].values).all()

n_use = min(len(best_weights), len(test_models))
test_models = test_models[:n_use]
test_model_names = test_model_names[:n_use]
best_weights = best_weights[:n_use]
s = sum(best_weights)
best_weights = [w / s for w in best_weights]

print("Blending models used for submission:", list(zip(test_model_names, best_weights)))

submission = test_models[0][["id"]].copy()
submission[TARGET] = 0.0
for i, t in enumerate(test_models):
    submission[TARGET] += t[PREDICTION].astype(float).values * best_weights[i]

if os.path.exists(SAMPLE_SUB_PATH):
    sample = pd.read_csv(SAMPLE_SUB_PATH, usecols=["id"])
    submission = sample.merge(submission, on="id", how="left")
    submission[TARGET] = submission[TARGET].fillna(global_mean)

print(submission.head())
print(submission.shape)

out_path = "submission.csv"
submission.to_csv(out_path, index=False)

print("Wrote", out_path, "with shape", submission.shape)
print(submission.dtypes)
print("Files in working dir:", os.listdir("."))

Blending models used for submission: [('type_mean', 0.009235170825816225), ('type_median', 0.0788168472143004), ('shrink_k1', 0.04350416861374703), ('shrink_k2', 0.01849733800241985), ('atompair_shrunk', 0.030908141050156247), ('type_dist_linear', 0.07050035687685118), ('mulliken_linear', 0.5490906176012332), ('shield_linear', 0.1994473598154758)]
        id  scalar_coupling_constant
0  2324604                142.472583
1  2324605                  0.675177
2  2324606                  1.910477
3  2324607                  1.910477
4  2324608                 -2.728967
(467813, 2)
Wrote submission.csv with shape (467813, 2)
id                            int64
scalar_coupling_constant    float64
dtype: object
Files in working dir: ['champs-scalar-coupling_ruhong_champs-scalar-coupling-blend_v1_C1.ipynb', 'champs-scalar-coupling', 'submission.csv']
